# Commuter Time Budget

Computes time budget for W-K-H (Work-Leisure-Home) trip chain accessibility.

**Output:** `dbs/data_p/commuter_time_budget.csv`
- `ID`: Individual identifier
- `time_budget`: Median weekday total travel time (minutes)
- `time_hw`: Median home-to-work commute time (minutes)
- `tt_wkh`: Remaining time for W-K-H detour = 90 - time_hw × 2
- `main_mode`: Main transport mode (for filtering in STA computation)

In [1]:
%cd D:\netmob25

D:\netmob25


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

## 1. Load trip data

In [3]:
df = pd.read_parquet('dbs/data_p/stays_extraction_all.parquet')
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])
df['date'] = df['trip_id'].str.split('_').str[0]

print(f'Total trips: {len(df)}')
print(f'Individuals: {df["ID"].nunique()}')

Total trips: 69706
Individuals: 3318


## 2. Identify workers

In [4]:
def is_worker(data):
    acts = data['purpose_o'].tolist() + data['purpose_d'].tolist()
    return 1 if 'WORK' in acts else 0

tqdm.pandas(desc='Identifying workers')
df_workers = (
    df.drop(columns=['ID'])
      .groupby(df['ID'])
      .progress_apply(is_worker)
      .reset_index(name='is_worker')
)

workers_id = df_workers[df_workers['is_worker'] == 1]['ID'].unique()
print(f'Workers: {len(workers_id)} out of {len(df_workers)} individuals')

Identifying workers: 100%|██████████| 3318/3318 [00:00<00:00, 12169.29it/s]


Workers: 2457 out of 3318 individuals


## 3. Compute commute time (home → work)

In [5]:
# Normalize purpose labels
df['purpose_d'] = df['purpose_d'].apply(lambda x: 'HOME' if 'HOME' in x else x)
df['purpose_o'] = df['purpose_o'].apply(lambda x: 'HOME' if 'HOME' in x else x)

# Filter to home→work trips
df_hw = df[(df['purpose_o'] == 'HOME') & (df['purpose_d'] == 'WORK')].copy()
df_hw = df_hw[df_hw['ID'].isin(workers_id)]

# Median commute time per individual
df_t_hw = df_hw.groupby('ID')['duration'].median().reset_index()
df_t_hw.columns = ['ID', 'time_hw']

print(f'Workers with commute data: {len(df_t_hw)}')
print(f'Median commute time: {df_t_hw["time_hw"].median():.1f} min')

Workers with commute data: 2294
Median commute time: 36.0 min


## 4. Compute total travel time budget

In [6]:
# Filter to weekdays only
df_weekday = df[df['dow'].isin(['monday', 'tuesday', 'wednesday', 'thursday', 'friday'])]
df_weekday = df_weekday[df_weekday['ID'].isin(workers_id)]

# Sum travel time per day, then median across days
df_tb = (
    df_weekday.groupby(['ID', 'date'])['duration'].sum()
    .reset_index()
    .groupby('ID')['duration'].median()
    .reset_index()
)
df_tb.columns = ['ID', 'time_budget']

print(f'Workers with travel budget data: {len(df_tb)}')
print(f'Median daily travel time: {df_tb["time_budget"].median():.1f} min')

Workers with travel budget data: 2457
Median daily travel time: 86.0 min


## 5. Compute remaining time for W-K-H trip chain

In [7]:
# Merge budget and commute time
df_time = pd.merge(df_tb, df_t_hw, on='ID', how='left')

# tt_wkh = 90 min fixed budget - round-trip commute
df_time['tt_wkh'] = 90 - df_time['time_hw'] * 2

# Load main_mode from attributes (for mode-specific STA filtering)
df_attr = pd.read_csv('dbs/data_p/commuter_attributes.csv', usecols=['ID', 'main_mode'])
df_attr = df_attr.drop_duplicates(subset=['ID'])

# Recode to Car vs PT (matching downstream analysis)
df_attr['is_car'] = df_attr['main_mode'].apply(
    lambda x: 1 if x in ['Private car (driver)', 'Private car (passenger)'] else 0
)

df_time = df_time.merge(df_attr[['ID', 'is_car']], on='ID', how='left')
df_time['is_car'] = df_time['is_car'].fillna(0).astype(int)

print(f'Individuals with positive tt_wkh: {(df_time["tt_wkh"] > 0).sum()} ({(df_time["tt_wkh"] > 0).mean()*100:.1f}%)')
print(f'Car users: {df_time["is_car"].sum()} ({df_time["is_car"].mean()*100:.1f}%)')
print(f'\nSummary:')
print(df_time[['time_budget', 'time_hw', 'tt_wkh']].describe().round(1))

Individuals with positive tt_wkh: 1428 (58.1%)
Car users: 914 (37.2%)

Summary:
       time_budget  time_hw  tt_wkh
count       2457.0   2294.0  2294.0
mean          94.8     40.5     9.0
std           49.7     25.1    50.3
min            5.0      1.5  -628.0
25%           60.0     22.0   -20.0
50%           86.0     36.0    18.0
75%          120.0     55.0    46.0
max          713.5    359.0    87.0


## 6. Save output

In [8]:
output_path = 'dbs/data_p/commuter_time_budget.csv'
df_time[['ID', 'time_budget', 'time_hw', 'tt_wkh', 'is_car']].to_csv(output_path, index=False)
print(f'Saved {len(df_time)} rows to {output_path}')

Saved 2457 rows to dbs/data_p/commuter_time_budget.csv


## 7. Save commuter trips (for downstream use)

In [9]:
df_commuter_trips = df[df['ID'].isin(workers_id)].copy()
trips_path = 'dbs/data_p/commuter_trips.csv'
df_commuter_trips.to_csv(trips_path, index=False)
print(f'Saved {len(df_commuter_trips)} trips to {trips_path}')

Saved 52965 trips to dbs/data_p/commuter_trips.csv
